# Ejercicio 1 - Selección y preparación de series MRI


## 1. Objetivo


Este notebook tiene como objetivo seleccionar una única serie de
resonancia magnética por estudio para la etapa posterior de
clasificación multietiqueta.

A partir de los 500 estudios con etiquetas débiles y de los 58 estudios
GOLD, se analiza la disponibilidad de planos y secuencias MRI. Se
selecciona como configuración principal una secuencia sagital sensible
a líquido con supresión de grasa, y posteriormente se resuelven los
casos que presentan múltiples series candidatas mediante metadatos DICOM.

El resultado final es un manifiesto reproducible con una única serie
seleccionada por estudio.

## 2. Carga de metadatos y subconjunto weak

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt

# Localizar la raíz del repositorio
actual = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        ruta for ruta in [actual, *actual.parents]
        if (ruta / "config" / "config.yaml").exists()
        and (ruta / "src").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, get_data_root

config = load_config()
DATA_ROOT = get_data_root(config)

TRAIN_SERIES_PATH = (
    DATA_ROOT / config["data"]["train_series_csv"]
)

TRAIN_PATH = (
    DATA_ROOT / config["data"]["train_csv"]
)

WEAK_PATH = (
    DATA_ROOT
    / config["data"]["processed"]["weak_labels"]
    / "weak_500_final.csv"
)

# csv: reproduce la selección desde el inventario, sin necesidad de DICOM.
# dicom: reconstruye el inventario leyendo las cabeceras DICOM.
MODO_INVENTARIO = "csv"

print("Modo de inventario:", MODO_INVENTARIO)
print("Proyecto:", PROJECT_ROOT)
print("Datos:", DATA_ROOT)

for nombre, ruta in {
    "train_series": TRAIN_SERIES_PATH,
    "train": TRAIN_PATH,
    "weak_500": WEAK_PATH
}.items():
    print(f"{nombre}: {ruta.exists()}")

    if not ruta.is_file():
        raise FileNotFoundError(ruta)

series = pd.read_csv(TRAIN_SERIES_PATH)

print("Dimensiones train_series:", series.shape)
display(series.head())

In [ ]:
weak_500 = pd.read_csv(WEAK_PATH)

print("Dimensiones:", weak_500.shape)
print(
    "Estudios únicos:",
    weak_500["StudyInstanceUID"].nunique()
)

display(
    weak_500[
        ["StudyInstanceUID", "Idioma"]
    ].head()
)

In [ ]:
series["StudyInstanceUID"] = (
    series["StudyInstanceUID"].astype(str)
)

weak_500["StudyInstanceUID"] = (
    weak_500["StudyInstanceUID"].astype(str)
)

## 3. Correspondencia entre estudios y series MRI

In [ ]:
series_500 = series[
    series["StudyInstanceUID"].isin(
        weak_500["StudyInstanceUID"]
    )
].copy()

print("=== CRUCE DE LOS 500 WEAK ===")

print(
    "Estudios encontrados:",
    series_500["StudyInstanceUID"].nunique()
)

print(
    "Series encontradas:",
    series_500["SeriesInstanceUID"].nunique()
)

faltantes = set(
    weak_500["StudyInstanceUID"]
) - set(
    series_500["StudyInstanceUID"]
)

print("Estudios faltantes:", len(faltantes))

In [ ]:
series_por_estudio = (
    series_500
    .groupby("StudyInstanceUID")
    ["SeriesInstanceUID"]
    .nunique()
)

display(
    series_por_estudio.describe()
)

## 4. Análisis de cobertura por plano y secuencia

Se evaluó la disponibilidad de las diferentes combinaciones de plano,
sensibilidad a líquido y supresión grasa tanto en los 500 estudios con
etiquetas débiles como en los 58 estudios GOLD.

El objetivo es seleccionar una secuencia con alta cobertura en ambos
conjuntos y adecuada para la clasificación multietiqueta.

In [ ]:
resumen_secuencias_500 = (
    series_500
    .groupby(
        [
            "Anatomical_Plane",
            "Fluid_Sensitive",
            "Fat_Suppression"
        ]
    )
    .agg(
        Series=("SeriesInstanceUID", "nunique"),
        Estudios=("StudyInstanceUID", "nunique")
    )
    .reset_index()
)

resumen_secuencias_500["Cobertura (%)"] = (
    100 *
    resumen_secuencias_500["Estudios"]
    / len(weak_500)
).round(1)

resumen_secuencias_500 = (
    resumen_secuencias_500
    .sort_values(
        ["Estudios", "Series"],
        ascending=False
    )
)

display(resumen_secuencias_500)

In [ ]:
train_path = TRAIN_PATH

train = pd.read_csv(train_path)

labels = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

gold_mask = train[labels].notna().all(axis=1)

gold = train[gold_mask].copy()

gold["StudyInstanceUID"] = (
    gold["StudyInstanceUID"].astype(str)
)

print("Estudios GOLD:", len(gold))



In [ ]:
series_gold = series[
    series["StudyInstanceUID"].isin(
        gold["StudyInstanceUID"]
    )
].copy()

print(
    "GOLD encontrados:",
    series_gold["StudyInstanceUID"].nunique()
)

print(
    "Series GOLD:",
    series_gold["SeriesInstanceUID"].nunique()
)

In [ ]:
resumen_secuencias_gold = (
    series_gold
    .groupby(
        [
            "Anatomical_Plane",
            "Fluid_Sensitive",
            "Fat_Suppression"
        ]
    )
    .agg(
        Series=("SeriesInstanceUID", "nunique"),
        Estudios=("StudyInstanceUID", "nunique")
    )
    .reset_index()
)

resumen_secuencias_gold["Cobertura (%)"] = (
    100 *
    resumen_secuencias_gold["Estudios"]
    / len(gold)
).round(1)

resumen_secuencias_gold = (
    resumen_secuencias_gold
    .sort_values(
        "Estudios",
        ascending=False
    )
)

display(resumen_secuencias_gold)

## 5. Selección de la secuencia principal

Se seleccionó como configuración principal:

**Sagittal + Fluid Sensitive + Fat Suppression**

Esta configuración está disponible en 490 de los 500 estudios con
etiquetas débiles (98.0 %) y en 56 de los 58 estudios GOLD (96.6 %).

La selección permite reducir considerablemente el volumen de imágenes
a procesar y mantener una cobertura elevada en ambos conjuntos.

In [ ]:
# ============================================================
# SECUENCIA PRINCIPAL
# Sagittal + Fluid Sensitive + Fat Suppression
# ============================================================

series_weak_sag = series_500[
    (series_500["Anatomical_Plane"] == "Sagittal") &
    (series_500["Fluid_Sensitive"] == 1) &
    (series_500["Fat_Suppression"] == 1)
].copy()

series_gold_sag = series_gold[
    (series_gold["Anatomical_Plane"] == "Sagittal") &
    (series_gold["Fluid_Sensitive"] == 1) &
    (series_gold["Fat_Suppression"] == 1)
].copy()

print("=== SAGITTAL FLUID-SENSITIVE + FAT-SUPPRESSION ===")

print(
    "Weak:",
    series_weak_sag["StudyInstanceUID"].nunique(),
    "/ 500 estudios"
)

print(
    "GOLD:",
    series_gold_sag["StudyInstanceUID"].nunique(),
    "/ 58 estudios"
)

print(
    "\nSeries weak:",
    series_weak_sag["SeriesInstanceUID"].nunique()
)

print(
    "Series GOLD:",
    series_gold_sag["SeriesInstanceUID"].nunique()
)

In [ ]:
multi_weak = (
    series_weak_sag
    .groupby("StudyInstanceUID")
    ["SeriesInstanceUID"]
    .nunique()
)

multi_gold = (
    series_gold_sag
    .groupby("StudyInstanceUID")
    ["SeriesInstanceUID"]
    .nunique()
)

print("=== WEAK ===")
print(multi_weak.value_counts().sort_index())

print("\nEstudios weak con más de una serie candidata:")
print((multi_weak > 1).sum())

print("\n=== GOLD ===")
print(multi_gold.value_counts().sort_index())

print("\nEstudios GOLD con más de una serie candidata:")
print((multi_gold > 1).sum())

In [ ]:
# ============================================================
# RUTA DICOM Y COMPROBACIÓN OPCIONAL
# ============================================================
TRAIN_DICOM_DIR = DATA_ROOT / "raw" / "sagittal"
DICOM_DISPONIBLES = TRAIN_DICOM_DIR.is_dir()

print("Ruta DICOM:", TRAIN_DICOM_DIR)
print("Directorio DICOM disponible:", DICOM_DISPONIBLES)

if DICOM_DISPONIBLES:
    estudios = [p for p in TRAIN_DICOM_DIR.iterdir() if p.is_dir()]
    series_dicom = [
        p for estudio in estudios
        for p in estudio.iterdir()
        if p.is_dir()
    ]
    print("Estudios en disco:", len(estudios))
    print("Series en disco:", len(series_dicom))
else:
    print("Modo CSV: no se requieren imágenes DICOM")

# Las 591 carpetas solo son obligatorias para reconstruir desde DICOM.
if MODO_INVENTARIO == "dicom":
    if not DICOM_DISPONIBLES:
        raise FileNotFoundError(TRAIN_DICOM_DIR)
    assert len(estudios) == 546, "Número inesperado de estudios DICOM"
    assert len(series_dicom) == 591, "Número inesperado de series DICOM"

# No incrustar imágenes DICOM en resultados destinados a GitHub.
# Para inspección local temporal, cambiar manualmente a True.
MOSTRAR_EJEMPLOS_DICOM = False


In [ ]:
MANIFEST_DIR = (
    DATA_ROOT
    / config["data"]["manifests"]["directory"]
)

INVENTORY_PATH = (
    MANIFEST_DIR
    / "inventory_sagittal_candidates.csv"
)

inventario_original = pd.read_csv(INVENTORY_PATH)

uids_originales = set(
    inventario_original.loc[
        inventario_original["Grupo"] == "WEAK",
        "StudyInstanceUID"
    ].astype(str)
)

uids_actuales = set(
    series_weak_sag["StudyInstanceUID"].astype(str)
)

print("Estudios WEAK originales:", len(uids_originales))
print("Estudios WEAK actuales:", len(uids_actuales))

print(
    "Actuales que no estaban originalmente:",
    len(uids_actuales - uids_originales)
)

print(
    "Originales que ya no aparecen:",
    len(uids_originales - uids_actuales)
)
assert uids_originales == uids_actuales, (
    "El conjunto WEAK actual no coincide con el inventario original. "
    "Comprueba que uses el weak_500_final.csv original de Kaggle."
)
print("✅ Cohorte WEAK original verificada")


### Estudios sin la secuencia seleccionada

La configuración **Sagittal + Fluid Sensitive + Fat Suppression**
no está disponible en 10 de los 500 estudios con etiquetas débiles
ni en 2 de los 58 estudios GOLD.

Por este motivo, el experimento principal basado en imágenes sagitales
utilizará **490 estudios WEAK y 56 estudios GOLD**.

Los estudios GOLD restantes no se eliminan del proyecto, ya que pueden
ser utilizados posteriormente en experimentos con otras secuencias o
planos que sí estén disponibles, como el plano axial.

## 6. Inventario de series DICOM candidatas

In [ ]:
# ============================================================
# INVENTARIO DE SERIES SAGITALES CANDIDATAS
# ============================================================

import os
import pandas as pd
import numpy as np
import pydicom

candidatas_sag = pd.concat([
    series_weak_sag.assign(Grupo="WEAK"),
    series_gold_sag.assign(Grupo="GOLD")
], ignore_index=True)

def inspeccionar_serie(row):

    study_uid = row["StudyInstanceUID"]
    series_uid = row["SeriesInstanceUID"]

    ruta = os.path.join(
        TRAIN_DICOM_DIR,
        study_uid,
        series_uid
    )

    archivos = [
        f for f in os.listdir(ruta)
        if f.lower().endswith(".dcm")
    ]

    n_slices = len(archivos)

    # Tamaño total de la serie
    bytes_total = sum(
        os.path.getsize(
            os.path.join(ruta, f)
        )
        for f in archivos
    )

    # Leer SOLO cabecera de un DICOM
    # No carga PixelData
    archivo_ejemplo = os.path.join(
        ruta,
        archivos[0]
    )

    ds = pydicom.dcmread(
        archivo_ejemplo,
        stop_before_pixels=True,
        force=True
    )

    return pd.Series({
        "Grupo": row["Grupo"],
        "StudyInstanceUID": study_uid,
        "SeriesInstanceUID": series_uid,
        "N_Slices": n_slices,
        "Tamaño_MB": bytes_total / (1024**2),
        "Rows": getattr(ds, "Rows", np.nan),
        "Columns": getattr(ds, "Columns", np.nan),
        "SliceThickness": getattr(
            ds,
            "SliceThickness",
            np.nan
        ),
        "SpacingBetweenSlices": getattr(
            ds,
            "SpacingBetweenSlices",
            np.nan
        ),
        "SeriesDescription": getattr(
            ds,
            "SeriesDescription",
            ""
        ),
        "ProtocolName": getattr(
            ds,
            "ProtocolName",
            ""
        ),
        "RepetitionTime": getattr(
            ds,
            "RepetitionTime",
            np.nan
        ),
        "EchoTime": getattr(
            ds,
            "EchoTime",
            np.nan
        )
    })


if MODO_INVENTARIO == "csv":

    print("Cargando inventario desde CSV...")
    inventario_sag = pd.read_csv(INVENTORY_PATH)

elif MODO_INVENTARIO == "dicom":

    print("Reconstruyendo inventario desde DICOM...")

    inventario_sag = candidatas_sag.apply(
        inspeccionar_serie,
        axis=1
    )

    ruta_reconstruido = (
        MANIFEST_DIR
        / "inventory_sagittal_rebuilt_local.csv"
    )

    inventario_sag.to_csv(
        ruta_reconstruido,
        index=False
    )

    print("Inventario reconstruido:", ruta_reconstruido)

else:
    raise ValueError("Modo no válido")

# Verificar las candidatas exactas
claves_esperadas = set(
    zip(
        candidatas_sag["StudyInstanceUID"].astype(str),
        candidatas_sag["SeriesInstanceUID"].astype(str)
    )
)

claves_inventario = set(
    zip(
        inventario_sag["StudyInstanceUID"].astype(str),
        inventario_sag["SeriesInstanceUID"].astype(str)
    )
)

assert len(inventario_sag) == 591
assert len(claves_inventario) == 591
assert claves_inventario == claves_esperadas

print("Series analizadas:", len(inventario_sag))
print("Estudios:", inventario_sag["StudyInstanceUID"].nunique())
print("Inventario validado correctamente")

display(inventario_sag.head())

In [ ]:
# ============================================================
# COMPROBACIÓN CRUZADA OPCIONAL: CSV VS DICOM
# ============================================================
# Esta validación se ejecuta cuando existe un inventario DICOM
# reconstruido previamente. No bloquea el modo CSV independiente.

INVENTARIO_DICOM_RECONSTRUIDO = (
    MANIFEST_DIR / "inventory_sagittal_rebuilt_local.csv"
)

inventario_csv = pd.read_csv(INVENTORY_PATH)
inventario_dicom = None

if INVENTARIO_DICOM_RECONSTRUIDO.is_file():
    inventario_dicom = pd.read_csv(INVENTARIO_DICOM_RECONSTRUIDO)

    claves = ["StudyInstanceUID", "SeriesInstanceUID"]
    csv_ordenado = (
        inventario_csv.sort_values(claves)
        .reset_index(drop=True).sort_index(axis=1)
    )
    dicom_ordenado = (
        inventario_dicom.sort_values(claves)
        .reset_index(drop=True).sort_index(axis=1)
    )

    print("Inventario CSV:", csv_ordenado.shape)
    print("Inventario DICOM:", dicom_ordenado.shape)

    pd.testing.assert_frame_equal(
        csv_ordenado,
        dicom_ordenado,
        check_dtype=False,
        check_exact=False,
        rtol=1e-5,
        atol=1e-4
    )
    print("✅ 591 series: inventarios CSV y DICOM equivalentes")
else:
    print("ℹ️ Modo CSV independiente: sin inventario DICOM reconstruido;")
    print("   se omite la comparación cruzada de metadatos.")


In [ ]:
print("=== NÚMERO DE CORTES ===")

display(
    inventario_sag["N_Slices"].describe()
)

print("\n=== TAMAÑO DE SERIES ===")

display(
    inventario_sag["Tamaño_MB"].describe()
)

print(
    "\nTamaño TOTAL de todas las series sagitales candidatas:",
    round(inventario_sag["Tamaño_MB"].sum() / 1024, 2),
    "GB"
)

## 7. Análisis de múltiples series por estudio

In [ ]:
conteo_candidatas = (
    inventario_sag
    .groupby("StudyInstanceUID")
    ["SeriesInstanceUID"]
    .nunique()
)

uids_duplicados = conteo_candidatas[
    conteo_candidatas > 1
].index

duplicados_sag = (
    inventario_sag[
        inventario_sag["StudyInstanceUID"].isin(
            uids_duplicados
        )
    ]
    .sort_values(
        [
            "StudyInstanceUID",
            "N_Slices"
        ]
    )
)

print(
    "Estudios con más de una serie:",
    len(uids_duplicados)
)

display(
    duplicados_sag[
        [
            "Grupo",
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "N_Slices",
            "Tamaño_MB",
            "Rows",
            "Columns",
            "SliceThickness",
            "SpacingBetweenSlices",
            "SeriesDescription",
            "ProtocolName"
        ]
    ].head(30)
)

In [ ]:
weak_sin_sag = weak_500[
    ~weak_500["StudyInstanceUID"].isin(
        series_weak_sag["StudyInstanceUID"]
    )
].copy()

gold_sin_sag = gold[
    ~gold["StudyInstanceUID"].isin(
        series_gold_sag["StudyInstanceUID"]
    )
].copy()

print(
    "WEAK sin Sagittal FS:",
    len(weak_sin_sag)
)

print(
    "GOLD sin Sagittal FS:",
    len(gold_sin_sag)
)

print("\nGOLD sin Sagittal FS:")
display(
    gold_sin_sag[
        ["StudyInstanceUID"] + labels
    ]
)

## 8. Clasificación y selección de una serie por estudio

Algunos estudios presentan más de una serie sagital compatible.
Por ello, se analizan las descripciones de las secuencias y parámetros
DICOM como `EchoTime` para caracterizar las adquisiciones y seleccionar
una única serie por estudio mediante una regla reproducible.

In [ ]:
# ============================================================
# CLASIFICAR SERIES SAGITALES SEGÚN DESCRIPCIÓN
# ============================================================

def clasificar_serie_sag(row):

    texto = (
        str(row["SeriesDescription"]) + " " +
        str(row["ProtocolName"])
    ).lower()

    # Detectar PD
    es_pd = any(
        x in texto
        for x in [
            " pd ",
            "pdw",
            "_pd",
            "pd_",
            "pd fs",
            "pd f/s",
            "pd fat",
            "proton"
        ]
    )

    # Detectar T2
    es_t2 = any(
        x in texto
        for x in [
            "t2",
            "t2w"
        ]
    )

    # Series que preferimos evitar si existe alternativa
    es_repeat = any(
        x in texto
        for x in [
            "repeat",
            "lower",
            "orig "
        ]
    )

    if es_pd:
        tipo = "PD"
    elif es_t2:
        tipo = "T2"
    else:
        tipo = "OTRO"

    return pd.Series({
        "Tipo_Secuencia": tipo,
        "Es_Repeat": es_repeat
    })


clasificacion = inventario_sag.apply(
    clasificar_serie_sag,
    axis=1
)

inventario_sag = pd.concat(
    [
        inventario_sag,
        clasificacion
    ],
    axis=1
)

print("=== TIPO DE SECUENCIA ===")

print(
    inventario_sag["Tipo_Secuencia"]
    .value_counts()
)

print("\n=== REPEAT / ORIG / LOWER ===")

print(
    inventario_sag["Es_Repeat"]
    .value_counts()
)

In [ ]:
print("=== DESCRIPCIONES MÁS FRECUENTES ===")

display(
    inventario_sag[
        [
            "SeriesDescription",
            "Tipo_Secuencia"
        ]
    ]
    .value_counts()
    .head(30)
)

In [ ]:
display(
    inventario_sag.sort_values(
        "N_Slices",
        ascending=False
    )[
        [
            "Grupo",
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "N_Slices",
            "Tamaño_MB",
            "Rows",
            "Columns",
            "SliceThickness",
            "SpacingBetweenSlices",
            "SeriesDescription",
            "ProtocolName",
            "RepetitionTime",
            "EchoTime"
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# COMPARAR ECHO TIME ENTRE PD, T2 Y OTROS
# ============================================================

display(
    inventario_sag
    .groupby("Tipo_Secuencia")[
        ["RepetitionTime", "EchoTime"]
    ]
    .describe()
)

In [ ]:
print("=== ECHO TIME POR TIPO ===")

for tipo in ["PD", "T2", "OTRO"]:

    subset = inventario_sag[
        inventario_sag["Tipo_Secuencia"] == tipo
    ]

    print(f"\n--- {tipo} ---")

    print(
        subset["EchoTime"]
        .describe()
    )

In [ ]:
print("\n=== OTROS ORDENADOS POR ECHO TIME ===")

display(
    inventario_sag[
        inventario_sag["Tipo_Secuencia"] == "OTRO"
    ][
        [
            "SeriesDescription",
            "RepetitionTime",
            "EchoTime",
            "N_Slices",
            "SliceThickness",
            "SpacingBetweenSlices"
        ]
    ]
    .sort_values("EchoTime")
    .head(40)
)

In [ ]:
display(
    inventario_sag[
        inventario_sag["Tipo_Secuencia"] == "OTRO"
    ][
        [
            "SeriesDescription",
            "RepetitionTime",
            "EchoTime",
            "N_Slices",
            "SliceThickness",
            "SpacingBetweenSlices"
        ]
    ]
    .sort_values(
        "EchoTime",
        ascending=False
    )
    .head(40)
)

### Clasificación de las secuencias

Las descripciones DICOM permitieron identificar directamente muchas
series PD y T2. Sin embargo, algunas series presentaban descripciones
genéricas o incompletas.

Para estos casos se utilizó `EchoTime` como criterio auxiliar. A partir
de la distribución observada en las series identificadas explícitamente,
se definieron las siguientes categorías:

- `PD`: identificación explícita de PD.
- `T2`: identificación explícita de T2.
- `PD_like`: descripción ambigua con TE ≤ 45 ms.
- `T2_like`: descripción ambigua con TE ≥ 50 ms.
- `AMBIGUO`: casos intermedios o sin información suficiente.

Esta clasificación se utiliza únicamente para seleccionar de forma
reproducible una serie por estudio y no representa una etiqueta clínica.

In [ ]:
# ============================================================
# CLASIFICACIÓN FINAL DE SERIES SAGITALES
# ============================================================

def clasificar_serie_final(row):

    texto = (
        str(row["SeriesDescription"]) + " " +
        str(row["ProtocolName"])
    ).lower()

    te = row["EchoTime"]

    # -------------------------
    # 1. Identificación explícita
    # -------------------------

    es_pd = (
        "pd" in texto
        or "proton" in texto
        or " dp " in f" {texto} "
    )

    es_t2 = (
        "t2" in texto
    )

    # -------------------------
    # 2. Evitar adquisiciones alternativas
    # -------------------------

    es_repeat = any(
        x in texto
        for x in [
            "repeat",
            "lower",
            "orig "
        ]
    )

    # -------------------------
    # 3. Clasificación
    # -------------------------

    if es_pd:
        tipo = "PD"

    elif es_t2:
        tipo = "T2"

    else:
        # Inferencia conservadora usando EchoTime
        if pd.notna(te):

            if te <= 45:
                tipo = "PD_like"

            elif te >= 50:
                tipo = "T2_like"

            else:
                tipo = "AMBIGUO"

        else:
            tipo = "AMBIGUO"

    return pd.Series({
        "Tipo_Final": tipo,
        "Es_Repeat_Final": es_repeat
    })


clasificacion_final = inventario_sag.apply(
    clasificar_serie_final,
    axis=1
)

inventario_sag = pd.concat(
    [
        inventario_sag,
        clasificacion_final
    ],
    axis=1
)

print("=== CLASIFICACIÓN FINAL ===")

print(
    inventario_sag["Tipo_Final"]
    .value_counts()
)

print("\nSeries alternativas/repetidas:")

print(
    inventario_sag["Es_Repeat_Final"]
    .value_counts()
)

In [ ]:
# ============================================================
# SELECCIONAR UNA SERIE SAGITAL POR ESTUDIO
# ============================================================

prioridad_tipo = {
    "PD": 1,
    "T2": 2,
    "PD_like": 3,
    "T2_like": 4,
    "AMBIGUO": 5
}

inventario_sag["Prioridad_Tipo"] = (
    inventario_sag["Tipo_Final"]
    .map(prioridad_tipo)
)

# Las series normales tienen prioridad
# sobre repeat/orig/lower
inventario_sag["Prioridad_Repeat"] = (
    inventario_sag["Es_Repeat_Final"]
    .astype(int)
)

seleccion_sag = (
    inventario_sag
    .sort_values(
        [
            "StudyInstanceUID",
            "Prioridad_Repeat",
            "Prioridad_Tipo",
            "SeriesInstanceUID"
        ]
    )
    .drop_duplicates(
        subset="StudyInstanceUID",
        keep="first"
    )
    .reset_index(drop=True)
)

print("=== SELECCIÓN SAGITAL FINAL ===")

print(
    "Estudios:",
    seleccion_sag["StudyInstanceUID"].nunique()
)

print("\nPor grupo:")
print(
    seleccion_sag["Grupo"]
    .value_counts()
)

print("\nTipo seleccionado:")
print(
    seleccion_sag["Tipo_Final"]
    .value_counts()
)

In [ ]:
# ============================================================
# VALIDACIÓN FINAL: SELECCIÓN POR AMBAS RUTAS
# ============================================================

def reproducir_seleccion(inventario):
    tabla = inventario.copy().reset_index(drop=True)
    clasificacion = tabla.apply(
        clasificar_serie_final, axis=1
    ).reset_index(drop=True)
    tabla = pd.concat([tabla, clasificacion], axis=1)
    tabla["Prioridad_Tipo"] = tabla["Tipo_Final"].map(prioridad_tipo)
    tabla["Prioridad_Repeat"] = tabla["Es_Repeat_Final"].astype(int)

    return (
        tabla.sort_values([
            "StudyInstanceUID", "Prioridad_Repeat",
            "Prioridad_Tipo", "SeriesInstanceUID"
        ])
        .drop_duplicates(subset="StudyInstanceUID", keep="first")
        .sort_values("StudyInstanceUID")
        .reset_index(drop=True)
    )

columnas_validacion = [
    "StudyInstanceUID", "SeriesInstanceUID", "Grupo", "Tipo_Final"
]

seleccion_csv = reproducir_seleccion(inventario_csv)[columnas_validacion]
seleccion_principal = (
    seleccion_sag[columnas_validacion]
    .sort_values("StudyInstanceUID")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    seleccion_csv, seleccion_principal, check_dtype=False
)
assert len(seleccion_csv) == 546
assert seleccion_csv["StudyInstanceUID"].nunique() == 546

print("✅ CSV y selección principal: las mismas 546 series")

if inventario_dicom is not None:
    seleccion_dicom = reproducir_seleccion(
        inventario_dicom
    )[columnas_validacion]
    pd.testing.assert_frame_equal(
        seleccion_csv, seleccion_dicom, check_dtype=False
    )
    print("✅ DICOM y CSV: idénticas 546 series seleccionadas")
else:
    print("ℹ️ Selección DICOM: comparación omitida (opcional)")

print("\nTipos de serie:")
print(seleccion_csv["Tipo_Final"].value_counts())
print("\nGrupo:")
print(seleccion_csv["Grupo"].value_counts())


In [ ]:
# ============================================================
# TAMAÑO DEL SUBCONJUNTO SAGITAL DEFINITIVO
# ============================================================

total_mb = seleccion_sag["Tamaño_MB"].sum()
total_gb = total_mb / 1024

print(
    "Series seleccionadas:",
    len(seleccion_sag)
)

print(
    "Tamaño total:",
    round(total_gb, 2),
    "GB"
)

print("\nTamaño por grupo:")

display(
    seleccion_sag
    .groupby("Grupo")["Tamaño_MB"]
    .agg(["count", "sum", "mean"])
    .assign(
        Total_GB=lambda x:
        x["sum"] / 1024
    )
)

In [ ]:
display(
    seleccion_sag[
        seleccion_sag["Tipo_Final"] == "AMBIGUO"
    ][
        [
            "Grupo",
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "SeriesDescription",
            "ProtocolName",
            "RepetitionTime",
            "EchoTime",
            "N_Slices",
            "SliceThickness"
        ]
    ]
)

## 9. Generación del manifiesto final

In [ ]:
columnas_manifest = [
    "Grupo",
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "Tipo_Final",
    "N_Slices",
    "Tamaño_MB",
    "Rows",
    "Columns",
    "SliceThickness",
    "SpacingBetweenSlices",
    "SeriesDescription",
    "ProtocolName",
    "RepetitionTime",
    "EchoTime"
]

manifest_sag = seleccion_sag[
    columnas_manifest
].copy()

MANIFEST_PATH = (
    MANIFEST_DIR
    / "manifest_sagittal_final.csv"
)

manifest_sag.to_csv(
    MANIFEST_PATH,
    index=False
)

# Verificación del archivo guardado
manifest_verificado = pd.read_csv(MANIFEST_PATH)

assert len(manifest_verificado) == 546
assert manifest_verificado["StudyInstanceUID"].nunique() == 546

print("✅ Manifest guardado correctamente")
print("Ruta:", MANIFEST_PATH)
print("Series:", len(manifest_verificado))

## 10. Validación visual de las series seleccionadas

In [ ]:
import pydicom
import numpy as np
import matplotlib.pyplot as plt
import os

def cargar_serie_dicom(study_uid, series_uid):

    ruta = os.path.join(
        TRAIN_DICOM_DIR,
        str(study_uid),
        str(series_uid)
    )

    archivos = [
        os.path.join(ruta, f)
        for f in os.listdir(ruta)
        if f.lower().endswith(".dcm")
    ]

    slices = []

    for archivo in archivos:

        ds = pydicom.dcmread(
            archivo,
            force=True
        )

        # Posición física del corte
        if (
            hasattr(ds, "ImagePositionPatient")
            and hasattr(ds, "ImageOrientationPatient")
        ):

            position = np.array(
                ds.ImagePositionPatient,
                dtype=float
            )

            orientation = np.array(
                ds.ImageOrientationPatient,
                dtype=float
            )

            row = orientation[:3]
            col = orientation[3:]

            normal = np.cross(row, col)

            posicion_corte = np.dot(
                position,
                normal
            )

        else:

            posicion_corte = float(
                getattr(
                    ds,
                    "InstanceNumber",
                    len(slices)
                )
            )

        slices.append(
            (
                posicion_corte,
                ds
            )
        )

    slices.sort(
        key=lambda x: x[0]
    )

    return [
        ds
        for _, ds in slices
    ]

In [ ]:
def dicom_a_imagen(ds):

    img = ds.pixel_array.astype(
        np.float32
    )

    slope = float(
        getattr(
            ds,
            "RescaleSlope",
            1.0
        )
    )

    intercept = float(
        getattr(
            ds,
            "RescaleIntercept",
            0.0
        )
    )

    img = (
        img * slope
        + intercept
    )

    # Normalización robusta
    p1, p99 = np.percentile(
        img,
        [1, 99]
    )

    img = np.clip(
        img,
        p1,
        p99
    )

    img = (
        img - p1
    ) / (
        p99 - p1 + 1e-8
    )

    return img

In [ ]:
def mostrar_serie(
    row,
    n_imagenes=9
):

    slices = cargar_serie_dicom(
        row["StudyInstanceUID"],
        row["SeriesInstanceUID"]
    )

    indices = np.linspace(
        0,
        len(slices) - 1,
        n_imagenes
    ).astype(int)

    fig, axes = plt.subplots(
        1,
        n_imagenes,
        figsize=(18, 4)
    )

    for ax, idx in zip(
        axes,
        indices
    ):

        img = dicom_a_imagen(
            slices[idx]
        )

        ax.imshow(
            img,
            cmap="gray"
        )

        ax.set_title(
            f"{idx}/{len(slices)-1}"
        )

        ax.axis("off")

    plt.suptitle(
        f"{row['Tipo_Final']} | "
        f"{len(slices)} cortes | "
        f"{row['SeriesDescription']}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
if MOSTRAR_EJEMPLOS_DICOM:
    for tipo in [
        "PD",
        "PD_like",
        "T2",
        "T2_like",
        "AMBIGUO"
    ]:

        candidatos = seleccion_sag[
            seleccion_sag["Tipo_Final"]
            == tipo
        ]

        if len(candidatos) > 0:

            print(
                f"\n=== {tipo} ==="
            )

            mostrar_serie(
                candidatos.iloc[0]
            )
else:
    print("ℹ️ Sin DICOM: se omiten imágenes de ejemplo")


In [ ]:
if MOSTRAR_EJEMPLOS_DICOM:
    serie_3d = (
        seleccion_sag
        .sort_values(
            "N_Slices",
            ascending=False
        )
        .iloc[0]
    )

    print(
        "Study:",
        serie_3d["StudyInstanceUID"]
    )

    print(
        "Cortes:",
        serie_3d["N_Slices"]
    )

    print(
        "Descripción:",
        serie_3d["SeriesDescription"]
    )

    mostrar_serie(
        serie_3d
    )
else:
    print("ℹ️ Sin DICOM: se omite la visualización 3D")


## 11. Conclusiones

A partir del subconjunto previamente construido de 500 estudios con
etiquetas débiles y de los 58 estudios GOLD, se analizaron las series
MRI disponibles mediante `train_series.csv`.

Se seleccionaron las series sagitales sensibles a líquido con supresión
de grasa como secuencia principal. Esta configuración presentó una
cobertura de 490/500 estudios weak y 56/58 estudios GOLD.

Debido a que algunos estudios contenían múltiples series candidatas,
se analizaron metadatos DICOM como `SeriesDescription`, `ProtocolName`,
`EchoTime`, número de cortes y parámetros espaciales. Se definió una
regla reproducible para seleccionar una única serie por estudio.

El resultado final fue un manifiesto de 546 estudios con una serie
sagital seleccionada por estudio:

- 490 estudios WEAK
- 56 estudios GOLD
- tamaño aproximado del subconjunto DICOM: 9.55 GB

Finalmente se realizó una inspección visual de ejemplos representativos
para verificar la coherencia anatómica y el orden de los cortes.

Este manifiesto será utilizado en la siguiente etapa para generar las
entradas 2.5D utilizadas por los modelos CNN.